# Nathan MODE 2Q - Backward / Adjoint Source-Scale Mask Synthesis

The displayed SLM/HWP masks are **not** the target; the propagated beam is the target, and the
masks are whatever the inverse optical system says they must be:

```text
V0 complex vector target at z_ref = 60 mm
  -> inverse angular-spectrum propagation (propagating modes only)
  -> inverse thin-axicon Jones (reciprocal p/s diagonal, conjugate conical phase)
  -> inverse (adjoint) QWP
  -> required H/V complex fields -> phase-only projected masks
  -> full forward verification through the source-scale bench operator
```

The 4F iris is treated as an adjoint projection (it discards frequencies; there is no exact
inverse) and its unreachable spectral energy is reported. No inherited objective/sample
microfabrication geometry, no MODE 1C/M1E constraints, no panel realism, and six bright lobes are
never called a hexagon unless the strict C3-vs-C6 classifier gate passes.

In [ ]:
from pathlib import Path

import vbb_study
from vbb_study.digital_twin import (
    run_mode2q_backward_mask_synthesis,
    write_mode2q_outputs,
)

PROJECT_ROOT = Path(vbb_study.__file__).resolve().parents[1]

In [ ]:
# Development scale (grid_n=512). The confirmation run below regenerates outputs at 1024.
report = run_mode2q_backward_mask_synthesis(grid_n=512, z_planes=61)
report["outcome"]["suggested_outcome"], report["outcome"]["outcome_statement"]

In [ ]:
# Backward-pass diagnostics: inverse consistency, amplitude realizability, 4F adjoint bookkeeping.
diag = dict(report["backward"].diagnostics)
{
    "complex_target_available": diag["target"]["complex_vector_target_available"],
    "recovery_overlap_to_raw_nathan": diag["recovery_vs_raw_nathan_input"]["overlap_to_raw_nathan_input"],
    "evanescent_clipped": diag["backpropagation"]["evanescent_clipped_energy_fraction"],
    "inverse_axicon_condition_number": diag["inverse_axicon"]["jones_condition_number"],
    "amp_H_mismatch_rms": diag["amplitude_vs_phase_only_supply"]["amp_H_over_supply_rms"],
    "amp_V_mismatch_rms": diag["amplitude_vs_phase_only_supply"]["amp_V_over_supply_rms"],
    "4f_kind": diag["four_f_adjoint"]["kind"],
    "4f_energy_outside_passband": diag["four_f_adjoint"]["required_field_energy_outside_passband_fraction"],
}

In [ ]:
# Forward-verified mask candidates (phase-only direct and through the carrier/4F chain).
list(report["candidate_rows"])

In [ ]:
paths = write_mode2q_outputs(
    output_dir=PROJECT_ROOT / "outputs" / "figures" / "digital_twin" / "nathan_mode2q_backward_mask_synthesis",
    grid_n=1024,
    z_planes=61,
)
{key: str(value) for key, value in paths.items()}

Expected outcome: `M2Q-A`. The backward pass recovers Nathan's raw pre-axicon field from the
V0 complex target (overlap ~1.0, inverse/forward operators consistent), the required H/V fields are
phase-only realizable (amplitude mismatch ~1e-4), and the inverse-designed masks reproduce the V0
hexagonal Bessel output after full forward propagation - directly and through the carrier/4F chain.
The low-dimensional optimiser exists but is skipped automatically because the analytic backward
initialisation already passes. This makes no microfabrication/sample-plane claim.